# AWS DynamoDB

## ClassicModels Orders (Denormalized Single-Table Design)

- DynamoDB is a **fully managed NoSQL** database provided by AWS, offering automatic scalability without manual intervention.

- This notebook migrates the **classicmodels** `orders` and `orderdetails` tables from RDS into a **single denormalized DynamoDB table** — the architecturally correct migration target — and contrasts every step with its SQL equivalent.

- Core idea: relational databases split related data across tables and JOIN at **read time**. DynamoDB flips this — you **pre-join at write time** by storing order-header and line-item fields together in one item, so every read is a cheap key lookup with no JOIN.

| Concept | RDS (MySQL) | DynamoDB |
|---------|-------------|----------|
| Schema | Fixed columns | Per-item (schemaless) |
| Relationships | Foreign keys / JOINs | Denormalise at write time |
| Query language | `SELECT … JOIN … WHERE` | Key-based `query()` + `FilterExpression` |
| Scaling | Vertical (+ read replicas) | Horizontal, automatic |
| Consistency | ACID transactions | Eventually consistent (default) |
| Efficient filter | Any indexed column | Must supply partition key |

https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/dynamodb.html


In [1]:
import boto3
import pymysql
import pymysql.cursors
import pandas as pd
from decimal import Decimal


## Connections

In [2]:
# Create a session using a specific profile
session = boto3.Session(profile_name='ashish')
credentials = session.get_credentials()

aws_access_key_id     = credentials.access_key
aws_secret_access_key = credentials.secret_key
region_name           = 'us-east-1'


In [3]:
# DynamoDB low-level client (used for table management)
dynamodb = boto3.client(
    'dynamodb',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region_name,
)

# DynamoDB resource (higher-level, used for put_item / query)
dynamodb_resource = boto3.resource(
    'dynamodb',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region_name,
)

print("DynamoDB client and resource ready.")


DynamoDB client and resource ready.


## Why Denormalize? – `orders` + `orderdetails`

In MySQL, order data lives in two normalised tables:

```
orders        (orderNumber PK, customerNumber FK, orderDate, status, comments)
orderdetails  (orderNumber FK, orderLineNumber, productCode, quantityOrdered, priceEach)
```

Every read that fetches an order with its line items requires a **JOIN** — a network round-trip + CPU cost that grows with data size.

### Benefits of denormalization for DynamoDB

| Benefit | Detail |
|---------|--------|
| **Single read = complete order** | One `query()` returns the header AND all line items — no second round-trip |
| **Data locality** | All items for a customer live on the same partition → predictable, low-latency reads |
| **Horizontal scalability** | Writes spread across partitions by `customerNumber`; no JOIN bottleneck as data grows |
| **Schema flexibility** | New fields (e.g., `discountCode`) can be added to individual items without `ALTER TABLE` |

### Tradeoffs to understand

| Tradeoff | Implication |
|----------|-------------|
| **Data duplication** | Order header fields (`orderDate`, `status`) repeat once per line item |
| **Update amplification** | Changing `status` requires updating every line-item record for that order |
| **No ad-hoc queries** | Filtering by non-key attributes (e.g., `productCode`) requires a slow `scan()` or a Global Secondary Index (GSI) |

### Table design

```
Table: ClassicModels_Orders
  Partition Key (PK):  customerNumber  (N)  ← "which customer placed this order?"
  Sort Key (SK):       SK              (S)  ← "ORDER#{orderNumber}#LINE#{lineNumber:02d}"
```

The sort-key prefix `ORDER#10100` groups all line items of order 10100 together, enabling a cheap prefix-range query with `begins_with`.  
A sample item looks like:

```json
{
  "customerNumber": 103,
  "SK": "ORDER#10100#LINE#01",
  "orderNumber": 10100,
  "orderLineNumber": 1,
  "orderDate": "2003-01-06",
  "status": "Shipped",
  "productCode": "S18_1749",
  "quantityOrdered": 30,
  "priceEach": 136.00,
  "lineTotal": 4080.00
}
```


In [4]:
def create_table_if_not_exists(table_name, key_schema, attribute_definitions):
    """Create a DynamoDB table; skip if it already exists."""
    existing = dynamodb.list_tables()['TableNames']
    if table_name in existing:
        print(f"Table '{table_name}' already exists – skipping creation.")
        return

    dynamodb.create_table(
        TableName=table_name,
        KeySchema=key_schema,
        AttributeDefinitions=attribute_definitions,
        BillingMode='PAY_PER_REQUEST',
    )
    waiter = dynamodb.get_waiter('table_exists')
    waiter.wait(TableName=table_name)
    print(f"Table '{table_name}' created successfully.")


# Single denormalized table – PK = customerNumber, SK = ORDER#{orderNumber}#LINE#{lineNumber}
create_table_if_not_exists(
    table_name='ClassicModels_Orders',
    key_schema=[
        {'AttributeName': 'customerNumber', 'KeyType': 'HASH'},
        {'AttributeName': 'SK',             'KeyType': 'RANGE'},
    ],
    attribute_definitions=[
        {'AttributeName': 'customerNumber', 'AttributeType': 'N'},
        {'AttributeName': 'SK',             'AttributeType': 'S'},
    ],
)

tbl_orders = dynamodb_resource.Table('ClassicModels_Orders')


Table 'ClassicModels_Orders' created successfully.


## Connect to RDS (pymysql)

Open a pymysql connection to the classicmodels database — this is the **source** for the migration.  
Update the credentials to match your own RDS instance (same values as in `2.2-AWS-RDS.ipynb`).


In [5]:
# RDS endpoint  (console → RDS → Databases → your instance → Connectivity → Endpoint)
DB_HOST     = "database-1.cvkgi402g6iv.us-east-1.rds.amazonaws.com"
DB_PORT     = 3306
DB_USER     = "admin"           # master username set when you created the instance
DB_PASSWORD = "dzYNcYDfTI2hgz4FR6F3"   # store in Secrets Manager or an env var in production
DB_NAME     = "classicmodels"

rds_conn = pymysql.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
    cursorclass=pymysql.cursors.DictCursor,
    autocommit=True,
)
print("Connected to RDS:", rds_conn.get_server_info())


Connected to RDS: 8.4.8


In [6]:
def fetch_rds(sql):
    """Run a query on RDS and return a list of dicts."""
    with rds_conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchall()


def to_dynamo_types(row: dict) -> dict:
    """
    Convert Python types so they are safe for DynamoDB:
    - float  → Decimal  (DynamoDB rejects native float)
    - None   → omitted  (DynamoDB has no NULL in put_item)
    - date   → str      (DynamoDB has no date type)
    """
    import datetime
    clean = {}
    for k, v in row.items():
        if v is None:
            continue
        if isinstance(v, float):
            clean[k] = Decimal(str(v))
        elif isinstance(v, (datetime.date, datetime.datetime)):
            clean[k] = str(v)
        else:
            clean[k] = v
    return clean


def denormalize_orders(orders_raw, details_raw):
    """
    Merge orders + orderdetails into a flat list ready for DynamoDB.

    Each DynamoDB item represents ONE order line item and carries:
    - all order-header fields (orderDate, status, comments …)
    - the specific line-item fields (productCode, quantityOrdered, priceEach)
    - a computed lineTotal

    By repeating the header fields per line item we eliminate JOINs at read time.
    """
    order_headers = {r['orderNumber']: r for r in orders_raw}
    items = []
    for detail in details_raw:
        order = order_headers[detail['orderNumber']]
        merged = {
            # Keys
            'customerNumber':  order['customerNumber'],
            'SK': f"ORDER#{order['orderNumber']}#LINE#{detail['orderLineNumber']:02d}",
            # Order header (denormalised – same value repeated per line item)
            'orderNumber':     order['orderNumber'],
            'orderDate':       order['orderDate'],
            'requiredDate':    order['requiredDate'],
            'shippedDate':     order['shippedDate'],   # None if not yet shipped
            'status':          order['status'],
            'comments':        order['comments'],
            # Line item
            'orderLineNumber': detail['orderLineNumber'],
            'productCode':     detail['productCode'],
            'quantityOrdered': detail['quantityOrdered'],
            'priceEach':       detail['priceEach'],
            'lineTotal':       round(detail['quantityOrdered'] * detail['priceEach'], 2),
        }
        items.append(to_dynamo_types(merged))
    return items


# ── Fetch from RDS ────────────────────────────────────────────────────────────
orders_raw  = fetch_rds("SELECT * FROM orders")
details_raw = fetch_rds("SELECT * FROM orderdetails")
print(f"Fetched {len(orders_raw)} orders and {len(details_raw)} line items from RDS.")

# ── Denormalize ───────────────────────────────────────────────────────────────
dynamo_items = denormalize_orders(orders_raw, details_raw)
print(f"Merged into {len(dynamo_items)} DynamoDB items ({len(dynamo_items) - len(orders_raw)} extra from denormalisation).")

# ── Batch-write to DynamoDB ───────────────────────────────────────────────────
with tbl_orders.batch_writer() as batch:
    for item in dynamo_items:
        batch.put_item(Item=item)

print(f"Loaded {len(dynamo_items)} items into ClassicModels_Orders.")


Fetched 326 orders and 2996 line items from RDS.
Merged into 2996 DynamoDB items (2670 extra from denormalisation).
Loaded 2996 items into ClassicModels_Orders.


## DynamoDB Query Patterns

Three fundamental read operations, ordered by cost:

| Operation | When to use | Cost |
|-----------|-------------|------|
| `get_item` | Exact PK + SK match — one item | Cheapest – O(1) |
| `query` | All items sharing a PK (+ optional SK range) | Cheap – reads only your partition |
| `scan` | Filter across the whole table | Expensive – reads every item first |

### Pattern – `query()` all orders for a customer (the primary access pattern)

SQL equivalent: `SELECT … FROM orders JOIN orderdetails … WHERE customerNumber = 103`

In DynamoDB you supply only the partition key — all matching items stream back in sort-key order, already merged (no JOIN needed because we denormalised at write time).


In [7]:
from boto3.dynamodb.conditions import Key, Attr

# ── DynamoDB: query all line items for customer 103 ──────────────────────────
response = tbl_orders.query(
    KeyConditionExpression=Key('customerNumber').eq(103)
)
df_ddb_cust = pd.DataFrame(response['Items'])
print(f"DynamoDB – customer 103: {len(df_ddb_cust)} line items across "
      f"{df_ddb_cust['orderNumber'].nunique()} orders")
df_ddb_cust[['orderNumber', 'orderDate', 'status', 'productCode',
             'quantityOrdered', 'priceEach', 'lineTotal']].head(10)


DynamoDB – customer 103: 7 line items across 3 orders


,orderNumber,orderDate,status,productCode,quantityOrdered,priceEach,lineTotal
0,10123,2016-05-20,Shipped,S24_1628,50,43.27,2163.5
1,10123,2016-05-20,Shipped,S18_1589,26,120.71,3138.46
2,10123,2016-05-20,Shipped,S18_2870,46,114.84,5282.64
3,10123,2016-05-20,Shipped,S18_3685,34,117.26,3986.84
4,10298,2014-09-27,Shipped,S10_2016,39,105.86,4128.54
5,10298,2014-09-27,Shipped,S18_2625,32,60.57,1938.24
6,10345,2014-11-25,Shipped,S24_2022,43,38.98,1676.14


## Clean Up

Drop both DynamoDB tables when the demo is done.  
Uncomment and run to avoid ongoing storage costs.


In [8]:
# for table_name in ('ClassicModels_Customers', 'ClassicModels_Offices'):
#     dynamodb.delete_table(TableName=table_name)
#     waiter = dynamodb.get_waiter('table_not_exists')
#     waiter.wait(TableName=table_name)
#     print(f"Table '{table_name}' deleted.")

rds_conn.close()
print("RDS connection closed.")


RDS connection closed.
